# Yawar Ñan — Tamizaje óptico de neutropenia grave en pediatría

**Hackatón Niño San Borja 2026 · Desafío 3: Ruta Hematológica**

Este notebook recorre el sistema completo, de la física a la decisión clínica:

1. El modelo óptico y por qué el umbral del adulto **no sirve** en un niño
2. Simulación física de videocapilaroscopía (para entrenar sin datos de pacientes)
3. El pipeline de visión, paso a paso y con figuras
4. Entrenamiento y validación del clasificador
5. Triaje clínico y salida interoperable (HL7 / FHIR)

> **Nota ética.** Ningún dato de este notebook proviene de pacientes reales. Las
> bases de la hackatón prohíben el uso de datos personales, y un tamizaje clínico
> tampoco puede sustentarse en "confíen en nosotros". La salida es simular el
> proceso físico completo, con verdad-terreno conocida por construcción.

In [ ]:
#@title Instalación  { display-mode: "form" }
# Única línea a ajustar: el repositorio del equipo.
REPO = "https://github.com/EQUIPO/REPOSITORIO.git"  #@param {type:"string"}

import os, sys, subprocess

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    destino = REPO.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(destino):
        subprocess.run(["git", "clone", "-q", REPO, destino], check=False)
    if os.path.exists(destino):
        os.chdir(destino)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "opencv-python-headless", "scikit-learn", "scipy"], check=False)

sys.path.insert(0, "src")
import numpy as np, matplotlib.pyplot as plt
import yawar
print("Yawar Ñan", yawar.__version__, "· directorio:", os.getcwd())

## 1. El modelo óptico

Bajo luz de ~420 nm (banda de Soret de la hemoglobina) los eritrocitos absorben
fuertemente y el capilar se ve oscuro. Un leucocito no tiene hemoglobina: deja
pasar la luz y desplaza a los eritrocitos aguas abajo. Aparece un **gap óptico**
brillante que viaja por el capilar.

Contar gaps es contar leucocitos. El modelo es puramente geométrico:

$$R = C \cdot v \cdot \pi (d/2)^2 \cdot 60 \cdot 10^{-9}$$

con $R$ en eventos/capilar/min, $C$ en células/µL, $v$ en µm/s y $d$ en µm.

Primero: ¿reproduce el modelo los valores publicados?

In [ ]:
from yawar.optics import wbc_from_event_rate, event_rate_from_wbc

# Bourquard et al., Sci Rep 2018 (PMC5871877), asumiendo v=800 µm/s y d=15 µm
print(f"32 eventos/min -> {wbc_from_event_rate(32):7.1f} células/µL   (paper: 3773)")
print(f" 2 eventos/min -> {wbc_from_event_rate(2):7.1f} células/µL   (paper:  236)")

### 1.1 El hallazgo que hace pediátrico al método

El método óptico cuenta **leucocitos totales**, no neutrófilos. Para llegar al
ANC hay que multiplicar por la fracción de neutrófilos — y en pediatría esa
fracción varía muchísimo con la edad. Entre el mes y los ~4 años hay predominio
linfocitario: sólo ~31% de los leucocitos son neutrófilos, frente a ~59% en el
adulto.

El dispositivo comercial de referencia usa un umbral fijo de ~7 gaps/min,
derivado de adultos. Veamos qué pasa al aplicarlo a un niño.

In [ ]:
from yawar.optics import (anc_from_wbc, neutrophil_fraction_for_age,
                          event_threshold_for_anc)

edades = [1, 2, 4, 6, 10, 16, 21]
print(f"{'edad':>5} {'frac neut':>10} {'ANC con umbral 7/min':>21} {'umbral correcto':>17}")
for e in edades:
    anc_7 = anc_from_wbc(wbc_from_event_rate(7.0), e)
    umbral = event_threshold_for_anc(500.0, e)
    marca = "  <-- SE PIERDE LA ALERTA" if anc_7 < 400 else ""
    print(f"{e:5} {neutrophil_fraction_for_age(e):10.2f} {anc_7:21.0f} {umbral:17.1f}{marca}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ed = np.linspace(0.5, 21, 200)

ax[0].plot(ed, [neutrophil_fraction_for_age(e) for e in ed], lw=2, color="#c1121f")
ax[0].set_xlabel("edad (años)"); ax[0].set_ylabel("fracción de neutrófilos")
ax[0].set_title("Neutrófilos / leucocitos totales")
ax[0].axvspan(1, 4, alpha=.12, color="#c1121f")
ax[0].annotate("predominio\nlinfocitario", (2.5, .36), ha="center", fontsize=9)
ax[0].grid(alpha=.3)

ax[1].plot(ed, [event_threshold_for_anc(500., e) for e in ed], lw=2,
           color="#003049", label="umbral pediátrico correcto")
ax[1].axhline(7.0, ls="--", color="#c1121f", label="umbral del adulto (7/min)")
ax[1].set_xlabel("edad (años)"); ax[1].set_ylabel("gaps/capilar/min para ANC=500")
ax[1].set_title("Umbral de alerta según la edad")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print("En un niño de 2 años, el umbral del adulto sólo se dispara cuando el ANC")
print(f"ya cayó a {anc_from_wbc(wbc_from_event_rate(7.0), 2):.0f}/µL. Se pierde toda la franja crítica 272–500.")

## 2. Simulación física

El simulador construye el vídeo desde primeros principios: tren de eritrocitos
en fila india, huecos leucocitarios con estadística de Poisson, transporte
pulsátil, absorción de Beer-Lambert sobre un cilindro, y una cámara con
desenfoque, viñeteado, temblor de mano y ruido de fotones.

In [ ]:
from yawar.synth import PatientState, CapillaryState, OpticalSetup, render_capture

setup = OpticalSetup(duration_s=20.0, fps=60.0)
sano  = render_capture(PatientState(age_years=8, anc_per_ul=3000),
                       CapillaryState(), setup, seed=7, with_video=True)
grave = render_capture(PatientState(age_years=8, anc_per_ul=250),
                       CapillaryState(), setup, seed=7, with_video=True)

fig, axes = plt.subplots(2, 2, figsize=(13, 5.5))
for fila, (cap, nombre) in enumerate([(sano, "ANC 3000 (normal)"),
                                      (grave, "ANC 250 (grave)")]):
    axes[fila, 0].imshow(cap.video[0], cmap="gray")
    axes[fila, 0].set_title(f"{nombre} — un fotograma"); axes[fila, 0].axis("off")
    axes[fila, 1].imshow(cap.kymograph[:600].T, aspect="auto", cmap="gray",
                         origin="lower")
    axes[fila, 1].set_title(f"kymograph — {cap.n_events_visible} gaps reales en 20 s")
    axes[fila, 1].set_xlabel("tiempo (fotogramas)"); axes[fila, 1].set_ylabel("posición (µm)")
plt.tight_layout(); plt.show()

## 3. El pipeline de visión

    vídeo → estabilizar → segmentar lumen → kymograph
          → medir velocidad → reproyectar al marco material → contar gaps

Las dos ideas que sostienen el método:

**Auto-calibración.** El trabajo de referencia *asume* v = 800 µm/s y d = 15 µm
para todos. Nosotros los medimos en el propio vídeo, porque los capilares
pediátricos son distintos y porque un error del 15% en el diámetro es un 30% de
error en el recuento.

**Marco material.** Un gap que se mueve es difícil de detectar. Pero si se conoce
la velocidad, se puede mirar la sangre *desde la sangre*: en la coordenada
$\xi = s - D(t)$ el gap está quieto y su estría diagonal se vuelve una línea
vertical. La SNR mejora en $\sqrt{n}$ con los fotogramas en que es visible.

In [ ]:
from yawar.vision import (stabilize, segment_capillary, extract_kymograph,
                          estimate_velocity, detect_events)
from yawar.vision.segment import fit_diameter_um

cap = sano
video_est, shifts, residual = stabilize(cap.video)
seg = segment_capillary(video_est, setup.um_per_px)
diam, r2_ajuste = fit_diameter_um(video_est, seg, setup.um_per_px)
kymo = extract_kymograph(video_est, seg, setup.um_per_px, setup.fps)
vel = estimate_velocity(kymo)
det = detect_events(kymo, vel)

print(f"diámetro   real {cap.capillary.diameter_um:6.1f} µm  ->  medido {diam:6.1f} µm  (R²={r2_ajuste:.2f})")
print(f"velocidad  real {cap.capillary.velocity_um_s:6.0f} µm/s ->  medida {vel.velocity_um_s:6.0f} µm/s")
print(f"gaps       real {cap.n_events_visible:6}      ->  detectados {det.n_events:6}")
print(f"movimiento residual tras estabilizar: {residual:.2f} px")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))

ax[0].imshow(seg.score_map, cmap="magma")
ax[0].plot(seg.centerline_px[:, 0], seg.centerline_px[:, 1], "c-", lw=1.5)
ax[0].set_title("segmentación: oscuridad × actividad"); ax[0].axis("off")

ax[1].imshow(kymo.data[:700].T, aspect="auto", cmap="gray", origin="lower")
ax[1].set_title("kymograph (las estrías diagonales son el flujo)")
ax[1].set_xlabel("fotograma"); ax[1].set_ylabel("posición (µm)")

p = det.projection
v = p.valid
ax[2].plot(p.xi_um[v], p.profile[v], lw=.8, color="#333")
ax[2].plot(det.positions_um, np.interp(det.positions_um, p.xi_um[v], p.profile[v]),
           "v", color="#c1121f", ms=7, label=f"{det.n_events} gaps")
ax[2].set_xlabel("coordenada material ξ (µm)"); ax[2].set_ylabel("intensidad")
ax[2].set_title("marco material: cada gap se cuenta una sola vez")
ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

### 3.1 El requisito de fps

La velocimetría sigue estructuras que viajan con la sangre. Los eritrocitos
tienen período espacial ~11 µm, y a 800 µm/s la sangre avanza 13.3 µm entre
fotogramas a 60 fps: **más de un período**. La textura fina queda aliaseada.

Esto fija un requisito de hardware que conviene conocer antes de comprar nada.

In [ ]:
#@title Barrido fps × ANC (tarda ~3 min)
EJECUTAR_BARRIDO = False  #@param {type:"boolean"}

if EJECUTAR_BARRIDO:
    filas = []
    for fps in [30, 60, 120, 240]:
        fila = []
        for anc in [3000, 600]:
            errs = []
            for s in range(3):
                st = OpticalSetup(duration_s=15.0, fps=fps)
                c = render_capture(PatientState(age_years=8, anc_per_ul=anc),
                                   CapillaryState(velocity_um_s=800.), st,
                                   seed=s, with_video=True)
                ve_, _, _ = stabilize(c.video)
                sg = segment_capillary(ve_, st.um_per_px)
                k = extract_kymograph(ve_, sg, st.um_per_px, st.fps)
                errs.append(abs(estimate_velocity(k).velocity_um_s - 800) / 800 * 100)
            fila.append(np.median(errs))
        filas.append((fps, *fila))
    print(f"{'fps':>5} {'err% ANC 3000':>14} {'err% ANC 600':>13}")
    for f, a, b in filas:
        print(f"{f:5} {a:14.1f} {b:13.1f}")
else:
    print("Resultado ya medido (mediana de 5 semillas, v real = 800 µm/s):\n")
    print("  fps | ANC 3000 | ANC 1500 | ANC 600 | ANC 200")
    print("   30 |    81.2% |    81.2% |   22.2% |   81.2%   <- inservible")
    print("   60 |     6.9% |     7.3% |    7.3% |   18.2%")
    print("  120 |     5.5% |     4.5% |    4.5% |   10.0%")
    print("  240 |     3.9% |     5.9% |    5.0% |    1.9%")
    print("\n=> >=60 fps es requisito. Y el error empeora cuando el ANC baja,")
    print("   es decir, justo en el caso crítico: de ahí la basal por paciente.")

## 4. El dataset

Cada fila es **un paciente** con 5 capilares, procesado por el pipeline
completo. Si no existe el archivo, se genera una cohorte pequeña aquí mismo.

Un punto importante de metodología: la unidad de análisis es el paciente, no el
capilar. En el estudio de referencia el AUC pasa de 0.68 con un capilar a 1.00
con cinco. Validar por capilar metería los 5 capilares de un mismo niño en
pliegues distintos e inflaría el AUC artificialmente.

In [ ]:
from pathlib import Path
from yawar.model import FEATURE_NAMES, label_severe

RUTA = Path("data/cohorte.npz")
if RUTA.exists():
    d = np.load(RUTA, allow_pickle=True)
    X, anc_true = d["features"], d["anc_true"]
    print(f"Cargados {len(anc_true)} pacientes de {RUTA}")
else:
    print("Generando cohorte reducida (esto tarda unos minutos)...")
    import subprocess
    subprocess.run([sys.executable, "scripts/build_dataset.py",
                    "--n-patients", "120", "--duration", "30",
                    "--workers", "4", "--out", str(RUTA)], check=True)
    d = np.load(RUTA, allow_pickle=True)
    X, anc_true = d["features"], d["anc_true"]

y = label_severe(anc_true)
print(f"{len(y)} pacientes · {y.sum()} con neutropenia grave ({100*y.mean():.0f}%)")
print(f"{X.shape[1]} variables: {', '.join(FEATURE_NAMES[:5])}...")

## 5. Entrenamiento y validación

El modelo **corrige**, no adivina. La física ya entrega un ANC con unidades y
sentido; el modelo absorbe el sesgo sistemático del detector (que recupera
~70–86% de los eventos reales) y entrega una probabilidad calibrada.

El umbral operativo se elige por **sensibilidad objetivo**, no maximizando
exactitud: no detectar una neutropenia grave (un niño con fiebre que se queda en
casa) y detectarla de más (un viaje evitable) no son errores comparables.

In [ ]:
from yawar.model import cross_validate, YawarClassifier, evaluate

proba, met = cross_validate(X, y, anc_true, n_splits=5, target_sensitivity=0.95)

print(f"AUC                {met.auc:.3f}")
print(f"Sensibilidad       {met.sensitivity:.3f}")
print(f"Especificidad      {met.specificity:.3f}")
print(f"VPN                {met.npv:.3f}   <- lo que importa en tamizaje")
print(f"VPP                {met.ppv:.3f}")
print(f"Brier              {met.brier:.3f}   (calibración; menor es mejor)")
print(f"umbral operativo   {met.threshold:.3f}")

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

fpr, tpr, _ = roc_curve(y, proba)
ax[0].plot(fpr, tpr, lw=2, color="#003049", label=f"AUC = {met.auc:.3f}")
ax[0].plot([0, 1], [0, 1], "k--", alpha=.4)
ax[0].set_xlabel("1 - especificidad"); ax[0].set_ylabel("sensibilidad")
ax[0].set_title("ROC (validación cruzada por paciente)")
ax[0].legend(); ax[0].grid(alpha=.3)

frac_pos, media_pred = calibration_curve(y, proba, n_bins=8, strategy="quantile")
ax[1].plot(media_pred, frac_pos, "o-", color="#c1121f")
ax[1].plot([0, 1], [0, 1], "k--", alpha=.4)
ax[1].set_xlabel("probabilidad predicha"); ax[1].set_ylabel("frecuencia observada")
ax[1].set_title("Calibración"); ax[1].grid(alpha=.3)

ax[2].scatter(anc_true, np.expm1(X[:, 0]), c=proba, cmap="RdYlGn_r", s=18,
              edgecolor="k", linewidth=.3)
lim = [50, 7000]
ax[2].plot(lim, lim, "k--", alpha=.5, label="identidad")
ax[2].axvline(500, color="#c1121f", ls=":"); ax[2].axhline(500, color="#c1121f", ls=":")
ax[2].set_xscale("log"); ax[2].set_yscale("log")
ax[2].set_xlabel("ANC real (/µL)"); ax[2].set_ylabel("ANC estimado por física (/µL)")
ax[2].set_title("Estimación física vs verdad"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

### 5.1 ¿Hace falta el modelo? Comparación honesta

Antes de embarcar un modelo hay que justificar que supera a no tener ninguno.
La línea base es usar directamente el ANC físico como score.

El resultado nos hizo cambiar de arquitectura a mitad del desarrollo.

In [ ]:
from yawar.model import physics_only_auc, cross_validate

filas = [("física sola (score = -log ANC)", physics_only_auc(X, y), None)]
for etiqueta, kind, compact in [
    ("logística compacta (4 variables)", "logistica", True),
    ("logística sobre las 13 variables", "logistica", False),
    ("gradient boosting (13 variables)", "arbol", False),
]:
    _, m = cross_validate(X, y, anc_true, kind=kind, compact=compact)
    filas.append((etiqueta, m.auc, m.brier))

print(f"{'modelo':38} {'AUC':>6} {'Brier':>7}")
for nombre, auc, brier in filas:
    print(f"{nombre:38} {auc:6.3f} {brier if brier is None else f'{brier:7.3f}'}")

**El gradient boosting es peor que no usar modelo.** Con 300 pacientes y 13
variables sobreajusta y además destruye la calibración. La logística compacta
gana con cuatro parámetros.

Elegimos la versión de cuatro variables pese a que la de trece tiene un Brier
ligeramente mejor, porque el destino de este modelo es **reajustarse con 30–50
casos reales** del INSNSB. Con esa cantidad de datos, cuatro parámetros se
estiman y trece no.

Y hay una ventaja adicional: el modelo es auditable. Sus coeficientes tienen
signo físicamente correcto — redescubre por su cuenta que *concentración =
eventos / volumen*.

In [ ]:
clf_audit = YawarClassifier(target_sensitivity=0.95).fit(X, y, anc_true)
for k, v in clf_audit.coefficients().items():
    print(f"  {k:24} {v:+.3f}")
print("\n  más eventos -> menos probable neutropenia (signo negativo) ✓")
print("  más volumen con los mismos eventos -> menor concentración (positivo) ✓")

In [ ]:
#@title Entrenar el modelo final y guardarlo
clf = YawarClassifier(target_sensitivity=0.95).fit(X, y, anc_true)
clf.calibrate_threshold(proba, y)
clf.metrics_ = met
clf.save("models/yawar_clf.pkl")
print(f"Modelo guardado. Umbral operativo = {clf.threshold_:.3f}")
print(f"Corrección de ANC: log1p(ANC) = {clf.anc_correction_[0]:.3f} + {clf.anc_correction_[1]:.3f}·log1p(ANC_físico)")

## 6. De la probabilidad a la conducta

Un número no cambia el desenlace de un niño con LLA. Lo que lo cambia es qué se
hace en las dos horas siguientes.

Dos reglas que no son obvias:

- **La fiebre manda sobre el número.** Un tamizaje dudoso nunca puede rebajar la
  conducta que ya indica la clínica; sólo puede subirla. El equipo sirve para
  *detectar* riesgo, jamás para *descartarlo*.
- **Se decide sobre el límite inferior del intervalo**, no sobre la estimación
  puntual. Con pocos eventos el intervalo es ancho, y usar el centro sería
  fingir una precisión que no se tiene.

In [ ]:
from yawar.pipeline import analyze_clip, aggregate
from yawar.triage import ClinicalContext, triage

# Caso demo: niña de 6 años, día 10 post-quimioterapia, con fiebre
EDAD, ANC_REAL = 6.0, 380.0
paciente = PatientState(age_years=EDAD, anc_per_ul=ANC_REAL)
rng = np.random.default_rng(3)

mediciones = []
for k in range(5):
    c = CapillaryState(diameter_um=float(np.clip(rng.normal(14, 1.2), 8, 22)),
                       velocity_um_s=float(np.clip(rng.normal(800, 120), 300, 1600)),
                       visible_length_um=float(rng.uniform(150, 230)))
    cap_sim = render_capture(paciente, c, setup, seed=int(rng.integers(1e6)),
                             with_video=True)
    m = analyze_clip(cap_sim.video, setup.um_per_px, setup.fps,
                     prior_velocity_um_s=800.0)
    if m: mediciones.append(m)

res = aggregate(mediciones, EDAD)
ctx = ClinicalContext(age_years=EDAD, temperature_c=38.6, days_since_chemo=10,
                      hours_to_reference_center=9.0, has_central_line=True)
dec = triage(res, ctx)

print(f"ANC real {ANC_REAL:.0f} -> estimado {res.anc_estimate:.0f} "
      f"(IC95 {res.anc_ci_low:.0f}-{res.anc_ci_high:.0f})")
print(f"\n{'='*66}\n  SEMÁFORO: {dec.level.value.upper()} — {dec.title}\n{'='*66}")
print(f"Plazo:  {dec.timeframe}\nAcción: {dec.action}\n")
for r in dec.rationale:
    print(f"  · {r}")

## 7. Salida interoperable

El resultado sale en los dos formatos que el sistema peruano necesita: **HL7 v2**
para el HIS actual del INSNSB (Galenus) y **FHIR R4** para donde va la
interoperabilidad pública.

En ambos, el resultado viaja marcado como *tamizaje* y *preliminar*, con su
método explícito. Eso no es burocracia: es lo que impide que dentro de seis meses
alguien lea este valor en la historia clínica como si fuera un hemograma.

In [ ]:
from yawar.interop import build_oru_r01, build_bundle
import json

print("--- HL7 v2 (ORU^R01) ---")
print(build_oru_r01(res, dec, patient_id="INSNSB-DEMO-001").replace("\r", "\n"))

print("\n--- FHIR R4 (Bundle) ---")
bundle = build_bundle(res, dec, patient_id="INSNSB-DEMO-001", device_id="yawar-01")
print(f"{len(bundle['entry'])} recursos: "
      f"{', '.join(e['resource']['resourceType'] for e in bundle['entry'])}")
print(json.dumps(bundle["entry"][0]["resource"], indent=2, ensure_ascii=False)[:900] + " ...")

## 8. Qué queda por hacer

Lo honesto es decir dónde están los límites:

| Estado | Punto |
|---|---|
| ✅ | Modelo físico validado contra la literatura |
| ✅ | Pipeline completo, auto-calibrado, con control de calidad |
| ✅ | Corrección pediátrica por edad |
| ✅ | Triaje clínico e interoperabilidad HL7/FHIR |
| ⚠️ | **Entrenado sobre datos sintéticos.** Las métricas miden la coherencia del pipeline, no su exactitud clínica |
| ⚠️ | La amplitud de las ondas de densidad eritrocitaria (25%) es un supuesto a validar contra vídeo real |
| ❌ | Validación con vídeo real del INSNSB, contra hemograma como patrón de referencia |
| ❌ | Estudio de concordancia con un tamaño muestral que permita fijar el umbral definitivo |

El siguiente paso no es más código: es grabar unos pocos vídeos reales con
hemograma pareado. Con 30–50 pares, la capa de calibración se reajusta y las
métricas pasan a significar algo clínico.